# Notebook 7: CNNs — Image Classification
**Estimated time:** 40-50 min
**Prerequisites:** Notebooks 1-6

---
## What you will learn
1. Design a full CNN for 10-class image classification
2. Train on MNIST with a proper train/val/test split
3. Evaluate with accuracy and a confusion matrix
4. Apply data augmentation to improve generalization
5. Visualize what the network has learned

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Loading and Preprocessing MNIST

We'll use two separate transforms:
- **Training**: normalize + random augmentation
- **Test**: normalize only (deterministic, fair evaluation)

In [ ]:
train_transform = T.Compose([
    T.RandomRotation(10),           # rotate ± 10 degrees
    T.RandomAffine(0, translate=(0.1, 0.1)),  # small translation
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,))   # MNIST mean and std
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,))
])

# Download dataset
full_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=train_transform)
test_set   = torchvision.datasets.MNIST('./data', train=False, download=True, transform=test_transform)

# Split training into train + val
n_train = 50000
n_val   = 10000
train_set, val_set = random_split(full_train, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=0)

print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

## 2. CNN Architecture

Our architecture follows the classic pattern:
- Several **convolutional blocks** (Conv → BN → ReLU → Pool) to extract features
- **Global Average Pooling** to collapse spatial dimensions
- A small **classifier head** (fully connected layers)

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: (B, 1, 28, 28) -> (B, 32, 14, 14)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: (B, 32, 14, 14) -> (B, 64, 7, 7)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: (B, 64, 7, 7) -> (B, 128, 7, 7) [no pooling]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # -> (B, 128, 1, 1)
            nn.Flatten(),              # -> (B, 128)
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = MNISTClassifier().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

# Test forward pass
x = torch.randn(4, 1, 28, 28).to(DEVICE)
out = model(x)
print(f'Output shape: {out.shape}')  # (4, 10)

In [ ]:
# Training setup
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(X)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            total_loss += loss.item() * len(X)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(X)
    return total_loss / total, correct / total

EPOCHS = 15
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model,   val_loader,  criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    lr = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch+1:2d} | train {tr_loss:.4f}/{tr_acc:.4f} | val {vl_loss:.4f}/{vl_acc:.4f} | lr={lr:.5f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0.9, 1.0)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final test evaluation
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

## 3. Confusion Matrix

A confusion matrix shows which classes the model confuses with each other.
Row = true class, Column = predicted class. Diagonal = correct predictions.

In [ ]:
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE)
        preds = model(X).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(y)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Print per-class accuracy
for cls in range(10):
    cls_mask = all_labels == cls
    cls_acc  = (all_preds[cls_mask] == cls).mean()
    print(f'Digit {cls}: {cls_acc:.4f}')

In [ ]:
# Visualize some mistakes
mistake_idx = np.where(all_preds != all_labels)[0]
print(f'Total mistakes: {len(mistake_idx)}')

# Sample 10 mistakes
sample_idx = mistake_idx[:10]
test_data = torchvision.datasets.MNIST('./data', train=False, transform=test_transform)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, idx in zip(axes.flat, sample_idx):
    img, true_label = test_data[idx]
    pred_label = all_preds[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'True: {true_label}, Pred: {pred_label}', color='red')
    ax.axis('off')
plt.suptitle('Misclassified samples')
plt.tight_layout()
plt.show()

---
## YOUR TURN — Mini Project

**Task:** Apply this CNN to the **Fashion-MNIST** dataset (same format as MNIST but harder: 10 clothing categories).

**Steps:**
1. Load `torchvision.datasets.FashionMNIST` (same API as MNIST)
2. Add more aggressive augmentation to the training transform (try `RandomHorizontalFlip`, `RandomCrop`)
3. Train the same `MNISTClassifier` (or modify it with more filters) for 15 epochs
4. Plot confusion matrix — which clothing items are hardest to classify?
5. What's the test accuracy? Compare to MNIST — why is it lower?

**Classes:** T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

In [ ]:
# YOUR CODE HERE
# 1. Load FashionMNIST with augmentation

# 2. Reuse or modify MNISTClassifier

# 3. Training loop (reuse train_epoch / eval_epoch)

# 4. Confusion matrix

# 5. Analysis